# 01 Data Audit
Inspect what's cached on disk — raw match JSON, timeline JSON, rank meta, and the processed parquet.

In [ ]:
import sys
from pathlib import Path

# Make sure the src/ package is importable when running from notebooks/
repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from lolobj.config import RAW_DIR, PROCESSED_DIR
from lolobj.ingest import storage

print('repo root :', repo_root)
print('raw data  :', RAW_DIR)
print('processed :', PROCESSED_DIR)

## 1. Cached match counts

In [ ]:
match_ids = list(storage.iter_cached_match_ids())
print(f'{len(match_ids)} matches with both match + timeline JSON cached')
match_ids[:5]

## 2. Inspect one raw match

In [ ]:
import json

if not match_ids:
    print('No cached matches. Run seed_matches first.')
else:
    m = storage.load_match(match_ids[0])
    info = m.get('info', {})
    print('matchId      :', m['metadata']['matchId'])
    print('gameVersion  :', info.get('gameVersion'))
    print('gameDuration :', info.get('gameDuration'), 's')
    print('participants :', len(info.get('participants', [])))
    print('top-level keys:', list(m.keys()))
    print('info keys     :', list(info.keys()))

In [ ]:
# Show participant summary for the first match
if match_ids:
    participants = info.get('participants', [])
    for p in participants:
        print(f"  team={p['teamId']}  role={p.get('teamPosition','?'):<8}  "
              f"champ={p.get('championName','?'):<16}  puuid={p['puuid'][:12]}...")

## 3. Inspect one raw timeline

In [ ]:
if match_ids:
    tl = storage.load_timeline(match_ids[0])
    frames = tl.get('info', {}).get('frames', [])
    print(f'frames : {len(frames)}')
    print(f'last frame timestamp: {frames[-1]["timestamp"] / 60000:.1f} min' if frames else 'no frames')

    # Count event types
    from collections import Counter
    event_types = Counter(
        e.get('type') for f in frames for e in f.get('events', [])
    )
    print('\nEvent type counts:')
    for etype, count in event_types.most_common():
        print(f'  {etype:<40} {count}')

## 4. Rank meta (puuid -> bucket mapping)

In [ ]:
from collections import Counter

puuid_buckets = storage.load_puuid_buckets()
print(f'{len(puuid_buckets)} puuids with rank info')
print('Bucket distribution:', dict(Counter(puuid_buckets.values())))

## 5. Processed parquet overview

In [ ]:
import pandas as pd

parquet_path = PROCESSED_DIR / 'objective_windows.parquet'
if not parquet_path.exists():
    print('Parquet not found — run: python -m lolobj.features.objective_windows --sample')
else:
    df = pd.read_parquet(parquet_path)
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    df.head()

In [ ]:
# Quick dtypes + null check
if parquet_path.exists():
    print('Nulls per column:')
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print('\nDtypes:')
    print(df.dtypes)

In [ ]:
# Objective breakdown
if parquet_path.exists():
    print('Rows per objective:')
    print(df.groupby(['objective_type', 'objective_number']).size())
    print('\nRank buckets:')
    print(df['rank_bucket'].value_counts())
    print('\nPatches:')
    print(df['patch'].value_counts().head(10))

## 6. Rebuild parquet if needed
Run this cell to regenerate `objective_windows.parquet` from all cached matches.

In [ ]:
from lolobj.features.objective_windows import build_table_from_cache
from lolobj.features.objective_windows import _write_output
from lolobj.config import ensure_data_dirs

ensure_data_dirs()
rows = build_table_from_cache()
out_path = PROCESSED_DIR / 'objective_windows.parquet'
_write_output(rows, out_path)
print(f'Wrote {len(rows)} rows to {out_path}')

df = pd.read_parquet(out_path)
print(f'Shape: {df.shape}')
print('Rank buckets:', dict(df['rank_bucket'].value_counts()))